# TIMP2 Apo MD — 100 ns on Colab
**Purpose**: Extend the apo pocket stability simulation from 50 ns to 100 ns.  
**Hardware**: Google Colab GPU (T4/A100/H100)  
**Protocol**: Identical to local run — AMBER14/TIP3P-FB, 300 K, 1 bar, 0.15 M NaCl, 4 fs HMR  
**Output**: Trajectory + analysis JSON + publication figure  

Upload `1br9_fixed.pdb` from your local machine to Google Drive before running.


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
DRIVE_DIR = '/content/gdrive/My Drive/timp2_apo_100ns'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Output directory: {DRIVE_DIR}")

Mounted at /content/gdrive
Output directory: /content/gdrive/My Drive/timp2_apo_100ns


## 2. Install Dependencies

In [2]:
!pip install -q openmm pdbfixer mdanalysis matplotlib numpy
import openmm as mm
print(f"OpenMM: {mm.__version__}")
print(f"Platforms: {[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]}")
try:
    p = mm.Platform.getPlatformByName('CUDA')
    print("CUDA ready")
    PLATFORM = 'CUDA'
except:
    try:
        p = mm.Platform.getPlatformByName('OpenCL')
        print("OpenCL ready (CUDA unavailable)")
        PLATFORM = 'OpenCL'
    except:
        print("GPU unavailable, using CPU")
        PLATFORM = 'CPU'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 21.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.7 MB/s eta 0:00:00
OpenMM: 8.5.1
Platforms: ['Reference', 'CPU', 'OpenCL']
OpenCL ready (CUDA unavailable)


## 3. Upload Protein Structure
Upload `1br9_fixed.pdb` to your Google Drive `timp2_apo_100ns/` folder,
**or** run this cell to download fresh from RCSB and fix it.

In [3]:
import requests
from pdbfixer import PDBFixer
from openmm.app import PDBFile

PROTEIN_PATH = os.path.join(DRIVE_DIR, '1br9_fixed.pdb')

if os.path.exists(PROTEIN_PATH):
    print(f"Using existing: {PROTEIN_PATH}")
else:
    print("Downloading 1BR9 from RCSB...")
    raw = os.path.join(DRIVE_DIR, '1br9_raw.pdb')
    resp = requests.get("https://files.rcsb.org/download/1BR9.pdb", timeout=60)
    with open(raw, 'w') as f:
        f.write(resp.text)
    print("Fixing structure...")
    fixer = PDBFixer(filename=raw)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(True)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(PROTEIN_PATH, 'w'))
    print(f"Saved: {PROTEIN_PATH}")

pdb = PDBFile(PROTEIN_PATH)
print(f"Atoms: {pdb.topology.getNumAtoms()}, Residues: {sum(1 for r in pdb.topology.residues())}")

Using existing: /content/gdrive/My Drive/timp2_apo_100ns/1br9_fixed.pdb
Atoms: 3190, Residues: 257


## 4. Solvate and Create System

In [4]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import time

print("Setting up force field (AMBER14 + TIP3P-FB)...")
ff = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

print("Solvating (1.2 nm padding, 0.15 M NaCl)...")
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addSolvent(ff, model='tip3p', padding=1.2*nanometers,
                    ionicStrength=0.15*molar)
n_atoms = modeller.topology.getNumAtoms()
print(f"Solvated system: {n_atoms:,} atoms")

print("Creating system...")
system = ff.createSystem(modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds,
    hydrogenMass=1.5*amu)
print("Done!")

Setting up force field (AMBER14 + TIP3P-FB)...
Solvating (1.2 nm padding, 0.15 M NaCl)...
Solvated system: 214,236 atoms
Creating system...
Done!


## 5. Setup Platform and Minimize

In [5]:
platform = Platform.getPlatformByName(PLATFORM)

if PLATFORM == 'CUDA':
    properties = {'CudaPrecision': 'mixed', 'DeviceIndex': '0'}
elif PLATFORM == 'OpenCL':
    properties = {'Precision': 'mixed'}
else:
    properties = {}

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
simulation = Simulation(modeller.topology, system, integrator, platform, properties)
simulation.context.setPositions(modeller.positions)

print("Energy minimization...")
e0 = simulation.context.getState(getEnergy=True).getPotentialEnergy()
simulation.minimizeEnergy(maxIterations=10000)
e1 = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  Before: {e0}")
print(f"  After:  {e1}")

Energy minimization...
  Before: 9399028047.72267 kJ/mol
  After:  -4243030.78923581 kJ/mol


## 6. NVT + NPT Equilibration (400 ps total)

In [6]:
import sys

print("NVT equilibration (200 ps)...")
simulation.context.setVelocitiesToTemperature(300*kelvin)
simulation.reporters.append(
    StateDataReporter(sys.stdout, 25000, step=True, temperature=True,
                      potentialEnergy=True, speed=True))
t0 = time.time()
simulation.step(50000)  # 200 ps at 4 fs
print(f"  NVT done in {time.time()-t0:.0f}s")

print("\nNPT equilibration (200 ps)...")
system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
simulation.context.reinitialize(preserveState=True)
t0 = time.time()
simulation.step(50000)  # 200 ps at 4 fs
print(f"  NPT done in {time.time()-t0:.0f}s")

# Save checkpoint and starting structure
chk_path = os.path.join(DRIVE_DIR, 'equilibrated.chk')
start_path = os.path.join(DRIVE_DIR, 'md_start.pdb')
simulation.saveCheckpoint(chk_path)
state = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
PDBFile.writeFile(simulation.topology, state.getPositions(), open(start_path, 'w'))
print(f"Saved: {chk_path}")
print(f"Saved: {start_path}")
simulation.reporters = []

NVT equilibration (200 ps)...
#"Step","Potential Energy (kJ/mole)","Temperature (K)","Speed (ns/day)"
25000,-3620955.2636097614,300.858021005914,0
50000,-3620950.9763708618,301.3800911213426,163
  NVT done in 108s

NPT equilibration (200 ps)...
75000,-3630941.9880082663,300.7033608843003,149
100000,-3630429.0036577266,301.1100069062707,146
  NPT done in 124s
Saved: /content/gdrive/My Drive/timp2_apo_100ns/equilibrated.chk
Saved: /content/gdrive/My Drive/timp2_apo_100ns/md_start.pdb


## 7. Production MD (100 ns)

In [7]:
production_ns = 100
dt_fs = 4.0
nsteps = int(production_ns * 1e6 / dt_fs)
save_ps = 50.0
save_interval = int(save_ps * 1000 / dt_fs)
chk_interval = int(5.0 * 1e6 / dt_fs)  # every 5 ns
dcd_path = os.path.join(DRIVE_DIR, 'md_production.dcd')
log_path = os.path.join(DRIVE_DIR, 'production.log')
chk_path = os.path.join(DRIVE_DIR, 'production.chk')
simulation.reporters.append(DCDReporter(dcd_path, save_interval))
simulation.reporters.append(
    StateDataReporter(log_path, save_interval,
        step=True, time=True, temperature=True,
        potentialEnergy=True, totalEnergy=True,
        speed=True, remainingTime=True, totalSteps=nsteps))
simulation.reporters.append(CheckpointReporter(chk_path, chk_interval))
print(f"Production: {production_ns} ns, {nsteps:,} steps")
print(f"Save every {save_ps} ps ({nsteps//save_interval} frames)")
print(f"Checkpoint every 5 ns")
print()
chunk_ns = 10.0
chunk_steps = int(chunk_ns * 1e6 / dt_fs)
n_chunks = int(production_ns / chunk_ns)
t_start = time.time()
for i in range(n_chunks):
    simulation.step(chunk_steps)
    elapsed = time.time() - t_start
    done_ns = (i + 1) * chunk_ns
    speed = (done_ns / elapsed) * 86400
    remaining = (production_ns - done_ns) / speed * 24 if speed > 0 else 0
    print(f"  {done_ns:6.0f}/{production_ns} ns  |  {speed:.0f} ns/day  |  ETA {remaining:.1f}h")
total = time.time() - t_start
print(f"\nDone! Wall time: {total/3600:.1f}h, Speed: {(production_ns/total)*86400:.0f} ns/day")
print(f"Trajectory: {dcd_path}")

Production: 100 ns, 25,000,000 steps
Save every 50.0 ps (2000 frames)
Checkpoint every 5 ns

      10/100 ns  |  138 ns/day  |  ETA 15.7h
      20/100 ns  |  138 ns/day  |  ETA 13.9h
      30/100 ns  |  138 ns/day  |  ETA 12.2h
      40/100 ns  |  138 ns/day  |  ETA 10.4h
      50/100 ns  |  138 ns/day  |  ETA 8.7h
      60/100 ns  |  138 ns/day  |  ETA 6.9h
      70/100 ns  |  138 ns/day  |  ETA 5.2h
      80/100 ns  |  138 ns/day  |  ETA 3.5h
      90/100 ns  |  138 ns/day  |  ETA 1.7h
     100/100 ns  |  138 ns/day  |  ETA 0.0h

Done! Wall time: 17.4h, Speed: 138 ns/day
Trajectory: /content/gdrive/My Drive/timp2_apo_100ns/md_production.dcd


## 8. Pocket Stability Analysis

In [8]:
import MDAnalysis as mda
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import json

matplotlib.use('Agg')

print("Loading trajectory...")
u = mda.Universe(
    os.path.join(DRIVE_DIR, 'md_start.pdb'),
    os.path.join(DRIVE_DIR, 'md_production.dcd'))
print(f"  {len(u.trajectory)} frames, {u.trajectory.totaltime/1000:.1f} ns")

triad = [6, 100, 103]
site1 = [3, 4, 5, 6, 8, 11, 100, 101, 102, 103, 104, 105]
triad_sel = f"name CA and resid {' '.join(str(r) for r in triad)}"
site1_sel = f"name CA and resid {' '.join(str(r) for r in site1)}"
protein_sel = "name CA"
triad_ca = u.select_atoms(triad_sel)
site1_ca = u.select_atoms(site1_sel)
protein_ca = u.select_atoms(protein_sel)
print(f"  Triad CA: {len(triad_ca)}, Site1 CA: {len(site1_ca)}, Protein CA: {len(protein_ca)}")

# Reference
u.trajectory[0]
ref_site1 = site1_ca.positions.copy()
ref_protein = protein_ca.positions.copy()

times, d_6_100, d_6_103, d_100_103 = [], [], [], []
pocket_rmsds, protein_rmsds = [], []

print("Analyzing frames...")
for ts in u.trajectory:
    times.append(ts.time / 1000)
    pos = {}
    for atom in triad_ca:
        pos[atom.resid] = atom.position
    if len(pos) == 3:
        d_6_100.append(np.linalg.norm(pos[triad[0]] - pos[triad[1]]))
        d_6_103.append(np.linalg.norm(pos[triad[0]] - pos[triad[2]]))
        d_100_103.append(np.linalg.norm(pos[triad[1]] - pos[triad[2]]))
    pocket_rmsds.append(np.sqrt(np.mean(np.sum((site1_ca.positions - ref_site1)**2, axis=1))))
    protein_rmsds.append(np.sqrt(np.mean(np.sum((protein_ca.positions - ref_protein)**2, axis=1))))

times = np.array(times)
d_6_100, d_6_103, d_100_103 = np.array(d_6_100), np.array(d_6_103), np.array(d_100_103)
pocket_rmsds, protein_rmsds = np.array(pocket_rmsds), np.array(protein_rmsds)

def tri_area(a, b, c):
    s = (a+b+c)/2
    return np.sqrt(np.maximum(s*(s-a)*(s-b)*(s-c), 0))

areas = tri_area(d_6_100, d_6_103, d_100_103)
crystal = {'d_6_100': 11.98, 'd_6_103': 8.22, 'd_100_103': 7.61}
crystal_area = tri_area(crystal['d_6_100'], crystal['d_6_103'], crystal['d_100_103'])
area_pres = abs(np.mean(areas) - crystal_area) / crystal_area
cv = np.std(areas) / np.mean(areas)

print(f"\nRESULTS:")
print(f"  Val6-Leu100:  {np.mean(d_6_100):.2f} +/- {np.std(d_6_100):.2f} (crystal: 11.98)")
print(f"  Val6-Phe103:  {np.mean(d_6_103):.2f} +/- {np.std(d_6_103):.2f} (crystal: 8.22)")
print(f"  Leu100-Phe103: {np.mean(d_100_103):.2f} +/- {np.std(d_100_103):.2f} (crystal: 7.61)")
print(f"  Triangle area: {np.mean(areas):.2f} +/- {np.std(areas):.2f} (crystal: {crystal_area:.2f})")
print(f"  Area preservation: {(1-area_pres)*100:.0f}%")
print(f"  CV: {cv*100:.0f}%")

verdict = "STABLE" if area_pres < 0.20 and cv < 0.30 else ("DYNAMIC" if area_pres < 0.40 else "UNSTABLE")
print(f"  Verdict: {verdict}")

# Save JSON
results = {
    "simulation": {"length_ns": float(times[-1]), "n_frames": len(times),
        "force_field": "AMBER14-all + TIP3P-FB", "temperature_K": 300,
        "pressure_bar": 1.0, "ionic_strength_M": 0.15, "timestep_fs": 4.0, "hmr": True},
    "triad_distances": {
        "Val6_Leu100": {"mean": float(np.mean(d_6_100)), "std": float(np.std(d_6_100)), "crystal": 11.98},
        "Val6_Phe103": {"mean": float(np.mean(d_6_103)), "std": float(np.std(d_6_103)), "crystal": 8.22},
        "Leu100_Phe103": {"mean": float(np.mean(d_100_103)), "std": float(np.std(d_100_103)), "crystal": 7.61}},
    "pocket_area": {"mean": float(np.mean(areas)), "std": float(np.std(areas)),
        "crystal": float(crystal_area), "preservation": float(1-area_pres), "cv": float(cv)},
    "rmsd": {"pocket_mean": float(np.mean(pocket_rmsds)), "pocket_std": float(np.std(pocket_rmsds)),
        "protein_mean": float(np.mean(protein_rmsds)), "protein_std": float(np.std(protein_rmsds))},
    "verdict": verdict}

json_path = os.path.join(DRIVE_DIR, 'md_analysis.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nSaved: {json_path}")

/usr/local/lib/python3.12/dist-packages/MDAnalysis/topology/PDBParser.py:309: UserWarning: PDB file is missing resid information.  Defaulted to '1'
  warnings.warn("PDB file is missing resid information.  "


Loading trajectory...


/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


  2000 frames, 100.0 ns
  Triad CA: 3, Site1 CA: 12, Protein CA: 194
Analyzing frames...

RESULTS:
  Val6-Leu100:  11.84 +/- 0.87 (crystal: 11.98)
  Val6-Phe103:  8.47 +/- 0.67 (crystal: 8.22)
  Leu100-Phe103: 7.79 +/- 0.40 (crystal: 7.61)
  Triangle area: 32.84 +/- 3.40 (crystal: 30.95)
  Area preservation: 94%
  CV: 10%
  Verdict: STABLE

Saved: /content/gdrive/My Drive/timp2_apo_100ns/md_analysis.json


## 9. Publication Figure

In [9]:
fig = plt.figure(figsize=(14, 16))
gs = GridSpec(4, 2, figure=fig, hspace=0.35, wspace=0.3, height_ratios=[1, 1, 1, 0.8])
window = max(1, len(times) // 100)
def rmean(x, w):
    return np.convolve(x, np.ones(w)/w, mode='valid') if len(x) >= w else x

# Panel A: Triad distances
ax = fig.add_subplot(gs[0, :])
for data, label, cry, color in [
    (d_6_100, 'Val6-Leu100', 11.98, '#1f77b4'),
    (d_6_103, 'Val6-Phe103', 8.22, '#ff7f0e'),
    (d_100_103, 'Leu100-Phe103', 7.61, '#2ca02c')]:
    ax.plot(times, data, alpha=0.15, color=color, linewidth=0.3)
    ax.plot(times[window-1:], rmean(data, window), color=color, linewidth=2, label=label)
    ax.axhline(y=cry, color=color, linestyle='--', alpha=0.4)
ax.set_ylabel('Distance (A)')
ax.set_title('(A) Hydrophobic Triad Distances', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)
ax.set_xlim(0, times[-1])

# Panel B: Area
ax = fig.add_subplot(gs[1, :])
ax.fill_between(times, areas, alpha=0.15, color='steelblue')
ax.plot(times[window-1:], rmean(areas, window), color='navy', linewidth=2)
ax.axhline(y=crystal_area, color='red', linestyle='--', linewidth=1.5, label=f'Crystal ({crystal_area:.1f})', alpha=0.7)
ax.axhline(y=np.mean(areas), color='orange', linestyle=':', linewidth=1.5, label=f'MD mean ({np.mean(areas):.1f} +/- {np.std(areas):.1f})')
ax.set_ylabel('Area (A^2)')
ax.set_title('(B) Pocket Openness', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)
ax.set_xlim(0, times[-1])

# Panel C: RMSD
ax = fig.add_subplot(gs[2, :])
ax.plot(times[window-1:], rmean(pocket_rmsds, window), color='darkgreen', linewidth=2, label='Site 1 pocket')
ax.plot(times[window-1:], rmean(protein_rmsds, window), color='gray', linewidth=1.5, linestyle='--', label='Full protein')
ax.set_ylabel('RMSD (A)')
ax.set_xlabel('Time (ns)')
ax.set_title('(C) RMSD vs Starting Structure', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)
ax.set_xlim(0, times[-1])

# Panel D: Histogram
ax = fig.add_subplot(gs[3, 0])
ax.hist(areas, bins=60, color='steelblue', alpha=0.7, edgecolor='navy', linewidth=0.3, density=True)
ax.axvline(x=crystal_area, color='red', linestyle='--', linewidth=2, label='Crystal')
ax.axvline(x=np.mean(areas), color='orange', linestyle='--', linewidth=2, label='MD mean')
ax.set_xlabel('Area (A^2)')
ax.set_ylabel('Density')
ax.set_title('(D) Area Distribution', fontweight='bold')
ax.legend(fontsize=9)

# Panel E: Violins
ax = fig.add_subplot(gs[3, 1])
parts = ax.violinplot([d_6_100, d_6_103, d_100_103], positions=[1,2,3], showmeans=True, showmedians=True)
for pc, color in zip(parts['bodies'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    pc.set_facecolor(color)
    pc.set_alpha(0.5)
ax.scatter([1,2,3], [11.98, 8.22, 7.61], color='red', s=80, zorder=5, marker='D', label='Crystal')
ax.set_xticks([1,2,3])
ax.set_xticklabels(['V6-L100','V6-F103','L100-F103'])
ax.set_ylabel('Distance (A)')
ax.set_title('(E) Distance Distributions', fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

fig.suptitle(f'TIMP2 Apo Pocket Stability — {times[-1]:.0f} ns MD\n(AMBER14/TIP3P-FB, 300 K, 1 bar, 0.15 M NaCl)', fontsize=15, fontweight='bold', y=0.98)

png_path = os.path.join(DRIVE_DIR, 'pocket_stability.png')
pdf_path = os.path.join(DRIVE_DIR, 'pocket_stability.pdf')
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
plt.show()

print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")

Saved: /content/gdrive/My Drive/timp2_apo_100ns/pocket_stability.png
Saved: /content/gdrive/My Drive/timp2_apo_100ns/pocket_stability.pdf


## 10. Done!
All output files are saved to your Google Drive at `timp2_apo_100ns/`:
- `md_start.pdb` — reference structure
- `md_production.dcd` — trajectory
- `production.log` — energy/temperature log
- `md_analysis.json` — numerical results
- `pocket_stability.png` / `.pdf` — publication figure

Download and send the JSON + figure to update the manuscript.
